# Build table of recommended algorithms

This notebook loops through cities, downloads and calculates street and bike network properties, and decides based on those properties which BikeNetKit algorithm should be recommended.

Outputs a ranked list of algorithms in a csv file that make sense for each city, like this:

```
slug,gbn,gbn+,lbn,fbn,sb
copenhagen,0,0,0,1,1
losangeles,1,0,0,0,1
budapest,0,1,1,1,1
...
```

Recommend algorithms as follows (to be tweaked):
```
gbn: cov_ratio < 0.15
gbn+: cov_ratio < 0.15 and l_ratio > 0.025
lbn: gcc3 > 0 and gcc1+gcc2+gcc3 > 0.3 and cov_ratio >= 0.15
fbn: gcc1 > 0.8 and cov_ratio > 0.4
sb: True
```

where:
- `cov_ratio` is the buffered bike network's area fraction of the area of the whole buffered street network
- `l_ratio` is the bike network's length fraction of the length of the whole street network
- `gcc1`, `gcc2`, `gcc3` are the fractions of the bike network's lengths in its largest, second, and third largest connected components

## Parameters

In [ ]:
debug = False
%run -i "functions.py"
cities = {"andorra": "Andorra la Vella", "budapest": "Budapest"}
# cities = {"copenhagen": "copenhagen municipality"}
bufferlength = 500
cf_biketracks = ['["cycleway"~"track"]',
                  '["highway"~"cycleway"]',
                  '["highway"~"path"]["bicycle"~"designated"]',
                  '["cycleway:right"~"track"]',
                  '["cycleway:left"~"track"]',
                  '["cyclestreet"]',
                  '["highway"~"living_street"]'
                 ]

## Run

In [ ]:
rec_table = []
for cityid, city_name in cities.items():
    rec_this = {"cityid": cityid, "sb": 1, "gbn": 0, "gbn+": 0, "lbn": 0, "fbn": 0}
    if debug: # Draw location polygons and their holes
        location = ox.geocoder.geocode_to_gdf(city_name)
        location = fill_holes(extract_relevant_polygon(cityid, shapely.geometry.shape(location['geometry'][0])))
        try:
            color = cm.rainbow(np.linspace(0,1,len(location)))
            for poly,c in zip(location, color):
                plt.plot(*poly.exterior.xy, c = c)
                for intr in poly.interiors:
                    plt.plot(*intr.xy, c = "red")
        except:
            plt.plot(*location.exterior.xy)
        plt.show()
        
    # DOWNLOAD NETWORKS
    # All streets
    nodes_car, edges_car, g_undir_car = prepare_network(city_name, network_type='all_public')
    edges_car_buffer = edges_car.buffer(bufferlength)

    # Bike tracks
    nodes_bike, edges_bike, g_undir_bike = prepare_network(city_name, custom_filter=cf_biketracks)
    edges_bike_buffer = edges_bike.buffer(bufferlength)
    
    fig = plt.figure(figsize=(4,4))
    ax = fig.add_axes([0,0,1,1])
    edges_car_buffer.plot(ax=ax)
    edges_car.plot(ax=ax, color="k")
    edges_bike_buffer.plot(ax=ax, color="orange")
    edges_bike.plot(ax=ax, color="r")
    ax.set_title(city_name)

    # CALCULATE METRICS
    try:
        cov_ratio = edges_bike_buffer.union_all().area / edges_car_buffer.union_all().area
        l_ratio = sum(edges_bike.length) / sum(edges_car.length)
        # https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.components.connected_components.html#networkx.algorithms.components.connected_components
        S = [g_undir_bike.subgraph(c).copy() for c in nx.connected_components(g_undir_bike)]
        edges_bike_length = sum([l[-1] for l in g_undir_bike.edges.data('length')])
        gcc1 = sum([l[-1] for l in S[0].edges.data('length')]) / edges_bike_length
    except: # no bike infra
        cov_ratio = l_ratio = gcc1 = edges_bike_length = 0
    try:
        gcc2 = sum([l[-1] for l in S[1].edges.data('length')])
    except:
        gcc2 = 0
    try:
        gcc3 = sum([l[-1] for l in S[2].edges.data('length')])
    except:
        gcc3 = 0
    
    print("cov_ratio: " + str(cov_ratio))
    print("l_ratio: " + str(l_ratio))
    print("gcc1: " + str(gcc1))
    print("gcc2: " + str(gcc2))
    print("gcc3: " + str(gcc3))

    # RECOMMEND ALGORITHMS
    if cov_ratio < 0.15:
        rec_this["gbn"] = 1
    if cov_ratio < 0.15 and l_ratio > 0.025:
        rec_this["gbn+"] = 1
    if gcc3 > 0 and gcc1+gcc2+gcc3 > 0.3 and cov_ratio >= 0.15:
        rec_this["lbn"] = 1
    if gcc1 > 0.8 and cov_ratio > 0.4:
        rec_this["fbn"]
    
    rec_table.append(rec_this)

## Export data

In [ ]:
with open('rec_table.csv', 'w', newline='') as csvfile:
    fieldnames = ['cityid', 'gbn', 'gbn+', 'lbn', 'fbn', 'sb']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rec_table)